# 01 Synthetic Data Overview

This notebook introduces the synthetic EcomPolice dataset and verifies that the active workflow uses generated data only.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "src").exists() and (path / "data").exists():
            return path
    raise RuntimeError("Run this notebook from inside the EcomPolice project.")

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

CLEAN_DIR = PROJECT_ROOT / "data" / "synthetic" / "clean"
MESSY_DIR = PROJECT_ROOT / "data" / "synthetic" / "messy"
LABELS_DIR = PROJECT_ROOT / "data" / "synthetic" / "labels"
MANIFESTS_DIR = PROJECT_ROOT / "data" / "synthetic" / "manifests"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

def load_tables(base_dir: Path) -> dict[str, pd.DataFrame]:
    return {
        "customers": pd.read_csv(base_dir / "customers.csv"),
        "products": pd.read_csv(base_dir / "products.csv"),
        "orders": pd.read_csv(base_dir / "orders.csv"),
        "deliveries": pd.read_csv(base_dir / "deliveries.csv"),
        "returns": pd.read_csv(base_dir / "returns.csv"),
    }

clean_tables = load_tables(CLEAN_DIR)
customer_labels = pd.read_csv(LABELS_DIR / 'customer_ground_truth.csv')
order_labels = pd.read_csv(LABELS_DIR / 'order_ground_truth.csv')
print(PROJECT_ROOT)


/Users/harshitmadhavpachauri/Documents/ChatGPT/EcomPolice


## Why Synthetic Data?

The project studies return behaviour without using real customer records or the old Kaggle workflow. Scenario labels are generated assumptions for academic experimentation; they are not verified real-world fraud labels.


In [2]:
for name, df in clean_tables.items():
    display(pd.DataFrame({'column': df.columns, 'dtype': df.dtypes.astype(str).values}))
    print(f'{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head())


,column,dtype
0,Customer_ID,object
1,Full_Name,object
2,City,object
3,State_Region,object
4,Signup_Date,object
5,Age_Band,object
6,Gender,object
7,Loyalty_Tier,object


customers: 3,000 rows x 8 columns


,Customer_ID,Full_Name,City,State_Region,Signup_Date,Age_Band,Gender,Loyalty_Tier
0,CUST000000,Aryan Maharaj,Pune,Maharashtra,2023-07-17,25-34,Male,Silver
1,CUST000001,Udant Dewan,Mumbai,Maharashtra,2023-11-20,18-24,Female,Bronze
2,CUST000002,Gagan Sami,Surat,Gujarat,2023-07-29,35-44,Male,Bronze
3,CUST000003,Ayushman Chander,Pune,Maharashtra,2023-07-02,18-24,Female,Platinum
4,CUST000004,Viraj Tiwari,Indore,Madhya Pradesh,2023-04-16,18-24,Female,Bronze


,column,dtype
0,Product_ID,object
1,Category,object
2,Brand,object
3,Product_Name,object
4,List_Price_INR,float64


products: 500 rows x 5 columns


...

deliveries: 20,000 rows x 4 columns


,Order_ID,Dispatch_Date,Delivery_Date,Delivery_Status
0,ORD0000334,2024-01-02,2024-01-07,Delivered
1,ORD0000975,2024-01-02,2024-01-07,Delivered
2,ORD0001915,2024-01-03,2024-01-08,Delivered
3,ORD0002062,2024-01-02,2024-01-09,Delivered
4,ORD0002196,2024-01-02,2024-01-05,Delivered


,column,dtype
0,Order_ID,object
1,Return_Date,object
2,Return_Reason,object
3,Refund_Amount_INR,float64
4,Return_Status,object


returns: 20,000 rows x 5 columns


,Order_ID,Return_Date,Return_Reason,Refund_Amount_INR,Return_Status
0,ORD0000334,NaN,NaN,NaN,Not Returned
1,ORD0000975,2024-02-03,Damaged item,43026.02,Refunded
2,ORD0001915,NaN,NaN,NaN,Not Returned
3,ORD0002062,NaN,NaN,NaN,Not Returned
4,ORD0002196,NaN,NaN,NaN,Not Returned


## Data Dictionary

These fields are the main columns used in the following notebooks.


In [3]:
dictionary = pd.DataFrame([
    ('Customer_ID', 'Stable synthetic customer identifier'),
    ('Product_ID', 'Stable synthetic product identifier'),
    ('Order_ID', 'Stable synthetic order identifier and analytical grain'),
    ('Return_Status', 'Observed return outcome such as Not Returned, Returned, Refunded, or Exchange Issued'),
    ('Scenario_Label', 'Generator-defined behavioural scenario stored separately from features'),
    ('Simulated_Abuse_Indicator', 'Artificial label for future supervised experiments, not a real fraud finding'),
], columns=['Column', 'Meaning'])
display(dictionary)


,Column,Meaning
0,Customer_ID,Stable synthetic customer identifier
1,Product_ID,Stable synthetic product identifier
2,Order_ID,Stable synthetic order identifier and analytic...
3,Return_Status,"Observed return outcome such as Not Returned, ..."
4,Scenario_Label,Generator-defined behavioural scenario stored ...
5,Simulated_Abuse_Indicator,Artificial label for future supervised experim...


In [4]:
summary = []
for name, df in clean_tables.items():
    id_cols = [col for col in df.columns if col.endswith('_ID')]
    summary.append({'table': name, 'rows': len(df), 'columns': df.shape[1], 'id_columns': ', '.join(id_cols), 'missing_cells': int(df.isna().sum().sum())})
display(pd.DataFrame(summary))


,table,rows,columns,id_columns,missing_cells
0,customers,3000,8,Customer_ID,0
1,products,500,5,Product_ID,0
2,orders,20000,10,"Order_ID, Customer_ID, Product_ID",0
3,deliveries,20000,4,Order_ID,91
4,returns,20000,5,Order_ID,49602


In [5]:
customers, products, orders, deliveries, returns = [clean_tables[k] for k in ['customers','products','orders','deliveries','returns']]
checks = {
    'orders_have_valid_customers': orders['Customer_ID'].isin(customers['Customer_ID']).all(),
    'orders_have_valid_products': orders['Product_ID'].isin(products['Product_ID']).all(),
    'deliveries_have_valid_orders': deliveries['Order_ID'].isin(orders['Order_ID']).all(),
    'returns_have_valid_orders': returns['Order_ID'].isin(orders['Order_ID']).all(),
    'order_labels_for_all_orders': orders['Order_ID'].isin(order_labels['Order_ID']).all(),
}
display(pd.Series(checks, name='passed'))


orders_have_valid_customers     True
orders_have_valid_products      True
deliveries_have_valid_orders    True
returns_have_valid_orders       True
order_labels_for_all_orders     True
Name: passed, dtype: bool

In [6]:
return_summary = returns['Return_Status'].value_counts(dropna=False).rename_axis('Return_Status').reset_index(name='orders')
scenario_summary = order_labels['Scenario_Label'].value_counts().rename_axis('Scenario_Label').reset_index(name='orders')
display(return_summary)
display(scenario_summary)


,Return_Status,orders
0,Not Returned,16534
1,Returned,1876
2,Refunded,1236
3,Exchange Issued,354


,Scenario_Label,orders
0,NORMAL,14401
1,FREQUENT_LEGITIMATE_RETURNS,2183
2,MIXED_SUSPICIOUS_PATTERN,968
3,REPEATED_DEFECT_CLAIMS,916
4,HIGH_VALUE_RETURN_PATTERN,771
5,RETURN_WINDOW_PATTERN,761


## Return Status vs Scenario Labels

`Return_Status` is an event outcome in the return table. Scenario labels are generator-defined ground truth kept separately for future modelling experiments. A returned order is not automatically suspicious.


## Conclusion

The active data source is the synthetic entity set. The clean tables have stable identifiers, separate labels, and clear joins for later analysis.
